# Can we predict late deliveries?
Target: `late_delivery` (1 = delivered after promised date). Data: one row per consignment.

**Key point up front:** `claim_paid_eur` is compensation paid on a claim *settled for the consignment*. It is only known after delivery and nearly restates the target, so it is excluded from the model (see the leakage check below). Only information available at dispatch is used.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, RocCurveDisplay
from sklearn.inspection import permutation_importance
df = pd.read_csv('data.csv', parse_dates=['dispatched_on']).sort_values('dispatched_on').reset_index(drop=True)
print(df.shape, df.consignment.is_unique, df.isna().sum().sum(), 'missing')
df.describe(include='all').T

## 1. Target and basic structure

In [ ]:
print('Late rate:', df.late_delivery.mean().round(3))
print(df.groupby('carrier').late_delivery.agg(['mean','size']))
print(df.dispatched_on.min(), '->', df.dispatched_on.max())
df.groupby(df.dispatched_on.dt.to_period('M')).late_delivery.mean().plot(marker='o', title='Late rate by dispatch month', ylim=(0,None));

## 2. Leakage check: `claim_paid_eur`

In [ ]:
has_claim = df.claim_paid_eur > 0
print(df.groupby(has_claim).late_delivery.agg(['mean','size']))
print('AUC of claim_paid_eur alone:', round(roc_auc_score(df.late_delivery, df.claim_paid_eur),3))

Consignments with a paid claim are late about 90% of the time, and those without are never late. The claim is a consequence of lateness, settled after delivery, so it is unavailable at dispatch time. Dropped.

## 3. Features available at dispatch

In [ ]:
df['hour'] = df.dispatched_on.dt.hour
df['dow'] = df.dispatched_on.dt.dayofweek
df['weekend'] = (df.dow>=5).astype(int)
df['kg_per_parcel'] = df.weight_kg/df.parcels
num = ['distance_km','weight_kg','parcels','kg_per_parcel','hour','dow','weekend']
cat = ['carrier']
X, y = df[num+cat], df.late_delivery
fig,ax = plt.subplots(1,4,figsize=(16,3))
for a,col in zip(ax,['distance_km','weight_kg','hour','dow']):
    g = df.groupby(pd.qcut(df[col],8,duplicates='drop'), observed=True).late_delivery.mean()
    a.plot(range(len(g)), g.values, marker='o'); a.set_title(f'late rate vs {col} (octiles)'); a.set_ylim(0,None)
plt.tight_layout()

## 4. Models
Time-ordered validation: expanding-window splits (train on the past, test on the future), since the model would be used going forward.

In [ ]:
pre = ColumnTransformer([('n',StandardScaler(),num),('c',OneHotEncoder(),cat)])
models = {
 'baseline (prior)': DummyClassifier(strategy='prior'),
 'logistic': make_pipeline(pre, LogisticRegression(max_iter=1000)),
 'grad. boosting': make_pipeline(ColumnTransformer([('n','passthrough',num),('c',OneHotEncoder(),cat)]),
                                 HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=0)),
}
tscv = TimeSeriesSplit(n_splits=5)
rows, oof = [], {}
for name,m in models.items():
    p = np.full(len(df), np.nan)
    for tr,te in tscv.split(X):
        m.fit(X.iloc[tr], y.iloc[tr]); p[te] = m.predict_proba(X.iloc[te])[:,1]
    mask = ~np.isnan(p); oof[name]=(mask,p)
    rows.append(dict(model=name, ROC_AUC=roc_auc_score(y[mask],p[mask]), PR_AUC=average_precision_score(y[mask],p[mask]), Brier=brier_score_loss(y[mask],p[mask])))
print('Late rate in tested rows:', y[oof['logistic'][0]].mean().round(3))
res = pd.DataFrame(rows).set_index('model').round(3); res

In [ ]:
fig,ax=plt.subplots(figsize=(5,4))
for n,(mk,p) in oof.items(): RocCurveDisplay.from_predictions(y[mk],p[mk],name=n,ax=ax)
ax.set_title('Out-of-time ROC');

## 5. What drives it, and does the ranking help?

In [ ]:
last_tr, last_te = list(tscv.split(X))[-1]
m = models['grad. boosting'].fit(X.iloc[last_tr], y.iloc[last_tr])
pi = permutation_importance(m, X.iloc[last_te], y.iloc[last_te], scoring='roc_auc', n_repeats=10, random_state=0)
pd.Series(pi.importances_mean, index=num+cat).sort_values().plot.barh(title='Permutation importance (drop in AUC), last fold');

In [ ]:
mk,p = oof['grad. boosting']
t = pd.DataFrame({'y':y[mk],'p':p[mk]}); t['decile']=pd.qcut(t.p.rank(method='first'),10,labels=False)+1
cal = t.groupby('decile').agg(pred=('p','mean'), actual=('y','mean'))
cal.plot(marker='o', title='Calibration / lift by predicted-risk decile'); cal.round(3).T

## Conclusions (from a run of the code above)
- **Yes, but only modestly.** Out-of-time ROC AUC is ~0.68 for both logistic regression and gradient boosting (PR AUC ~0.35 vs ~0.19 base rate in the tested period). The "baseline" row's AUC (~0.43) is an artefact of pooling constant predictions across folds with different base rates; read it as 0.5.
- The simple logistic model is as good as boosting, and better calibrated: boosting over-predicts at the high end (top decile predicted ~0.63, actual ~0.42). Prefer logistic, or calibrate boosting.
- Risk rises with weight and distance (late rate ~9% in the shortest-distance quintile to ~36% in the longest). `coastal` is later (27%) than `north` (19%) and `express` (18%). Hour, day of week and weekend show no clear signal.
- Useful for ranking/prioritising: the top predicted-risk decile is late ~42% of the time, versus ~7% in the bottom decile. Not accurate enough to call individual consignments.

Caveats
- `claim_paid_eur` is excluded as post-outcome leakage; including it would give a deceptively strong but unusable model.
- No weather, route, depot load or promised-lead-time data, which are likely real drivers of lateness.
- Validation is out-of-time within a ~2-year window; performance may drift.